<a href="https://colab.research.google.com/github/hiiamlars/analytical_flexibility_llm_reviews/blob/main/data_generation/llm_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Setup

In [ ]:
# @title Dependencies

# Installation
!pip install pandas pyarrow -q
!pip install numpy -q
!pip install tqdm -q
!pip install openai -q
!pip install groq -q

# First-party installations
import itertools
import re
import json
import random
import time
from typing import Dict, Any, Tuple, Optional, Literal, List
from dataclasses import dataclass
import os

# Third-party installations
from google.colab import drive
import google.colab.userdata
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from openai import OpenAI
from openai import APIError
from groq import Groq
from google import genai
from google.genai import types

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.5 MB/s eta 0:00:00


In [ ]:
# @title Path definitions

# Mount GDrive
drive.mount('/content/drive')

# Set working directory
WORKING_DIR = "/content/drive/MyDrive/Analytical_flexibility_llm_reviews"

# Create working directory (if not already existing)
os.makedirs(WORKING_DIR, exist_ok=True)
print(f"Ensured working directory exists at: {WORKING_DIR}.")

# Define and create DATASET_DIR to load the paper content
DATASET_DIR = os.path.join(WORKING_DIR, "iclr/final")
os.makedirs(DATASET_DIR, exist_ok=True)
print(f"Ensured dataset directory exists at: {DATASET_DIR}.")

# Define and create the RAW_DIR for the llm reviews (and their logs)
RAW_DIR = os.path.join(WORKING_DIR, "llm_reviews")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Ensured raw directory exists at: {RAW_DIR}.")

# Define the full result path
RESULTS_PATH = os.path.join(RAW_DIR, "llm_reviews_results.parquet")

# Define the full log path
LOG_PATH = os.path.join(RAW_DIR, "llm_reviews_log.parquet")

In [ ]:
# @title Setup definitions

### --- Settings --- ###

SIMULATION_ACTIVE = True # SIMULATION_ACTIVE = False activates the API calls

# Define the number of reviews to generate per paper-parameter combo
NUM_REVIEWS_PER_PAPER = 1

### --- Parameters --- ###

# OpenAI
OPENAI_MODELS = [
    "gpt-5-mini-2025-08-07",
    "gpt-5-nano-2025-08-07",
    "gpt-4o-2024-11-20", # Changed to a newer model version
    "gpt-4o-mini-2024-07-18",
    "o4-mini-2025-04-16",
    "o3-mini-2025-01-31",
    "gpt-3.5-turbo-0125"
]

# Groq
GROQ_MODELS = [
    "llama-3.3-70b-versatile"
]

# Gemini
GEMINI_MODELS = [
    "gemini-3-flash-preview" # new model
]

# Models who can vary by reasoning effort
REASONING_MODELS = {
    "o4-mini-2025-04-16",
    "gpt-5-mini-2025-08-07",
    "o3-mini-2025-01-31",
    "gpt-5-nano-2025-08-07",
    "gemini-3-flash-preview"
}

# Define temperature parameters (as used by Cummins 2025)
TEMPS = [0.0, 0.5, 1.0, 1.5]

# Define reasoning parameters (as used by Cummins 2025)
REASONING = ["low", "high"]

# Define prompt style (CoT) (as used by Li et al. 2025)
CoT = [
    "",
    "Explain your thought process step by step."
]

# Define note taking (adopted from Robertson and Kayejo (2025))
NOTE_TAKING = [
    "Faithful",
    "Objective",
    "Evaluation"
]

# Create data class for parameter combinations
@dataclass
class ParamCombo:
    model: str
    temperature: Optional[float]
    reasoning_effort: Optional[Literal[*REASONING]]
    chain_of_thought: Literal[*CoT]
    note_taking: Literal[*NOTE_TAKING]

### --- Content --- ####

# Target string to remove metadata from the papers full text
HEADER_TO_REMOVE = "GROBID - A machine learning software for extracting information from scholarly documents"

# Note taking strategies/instructions, adopted from Robertson and Koyejo (2025)
NOTE_TAKING_FAITHFUL = f"Take detailed, accurate notes on the paper for an ICLR style review. Focus on precisely capturing the actual methods, results, and contributions without any distortion. Just output the notes."
NOTE_TAKING_OBJECTIVE = f"Take comprehensive notes on the paper's technical content, experimental design, and results. Document both strengths and limitations objectively. Just output the notes."
NOTE_TAKING_EVALUATION = f"Take meticulous notes covering all aspects of the paper including theoretical foundations, experimental methodology, results, and implications. Note specific equations, theorem numbers, and experimental details. Just output the notes."

# Review generation instruction, adopted from Robertson and Koyejo (2025)
REVIEW_GENERATION = f"""

Create an ICLR-style review following this specific structure:

# Summary Of The Paper
Summarize the paper’s main contributions, methodology, and findings.

# Strength And Weaknesses
Analyze the paper’s contributions based on your notes.

# Clarity, Quality, Novelty And Reproducibility
Evaluate based on your notes.

# Summary Of The Review
Provide a 2-3 sentence distillation of your overall assessment.

# Correctness
Rate on a scale of 1-5.

# Technical Novelty And Significance
Rate on a scale of 1-5.

# Empirical Novelty And Significance
Rate on a scale of 1-5.

Maintain a professional tone throughout. Base your review entirely on your reading notes.

"""

# Predefined placeholder strings for consistent error/status reporting
NOTE_TAKING_FAILURE = "ERROR: NOTE TAKING NOT SUCCESSFUL"
REVIEW_GENERATION_FAILURE = "ERROR: REVIEW GENERATION NOT SUCCESSFUL"
UNKNOWN_ERROR_STATE = "ERROR: UNKNOWN STATE REACHED"

### --- API/Client --- ###

MAX_RETRIES = 3 # Random
RETRY_DELAY = 1.5 # Random
SEED_VALUE = 45 # Random
MAX_TOKENS = 2000 # As Robertson and Kayejo (2025) (in their repo)

# OpenAI API-key
OPENAI_KEY = google.colab.userdata.get('OPENAI_API_KEY')

# Groq API-key
GROQ_KEY = google.colab.userdata.get('GROQ_API_KEY')

# Gemini API-key
GEMINI_API_KEY = google.colab.userdata.get('GEMINI_API_KEY')

# Groq URL
GROQ_URL ="https://api.groq.com/openai/v1"

In [ ]:
# @title Data import

# Read dataset_paper
dataset_paper = pd.read_parquet(os.path.join(DATASET_DIR, "dataset_paper.parquet"))

# Select target columns
paper_content = dataset_paper[['paper_id', 'abstract', 'parsed_text']].copy()

# Remove the HEADER_TO_REMOVE string from the 'parsed_text' column
paper_content['parsed_text'] = paper_content['parsed_text'].str.replace(HEADER_TO_REMOVE, '', regex=False).str.strip()

# Check paper_content
display(paper_content.head(3))
display(paper_content.shape)

# Prepare Dict for later applications
targets = paper_content.to_dict('records')

### Client

In [ ]:
class LLMClient:
    def __init__(self, seed: int, simulate: bool = True):
        """Initialize LLMClient with defaults"""
        self.simulate = simulate
        self.rng = random.Random(seed)

        # Set placeholders for real clients when simulate=False:
        self._openai = None
        self._groq_openai_client = None
        self._gemini_client = None

    def _maybe_init_clients(self):
        """Start simulation/API clients"""
        if self.simulate:
            return
        # Initialize OpenAI client
        if self._openai is None and "OPENAI_KEY" in globals():
            self._openai = OpenAI(api_key=OPENAI_KEY)

        # Initialize Groq client using OpenAI client interface
        if self._groq_openai_client is None and "GROQ_KEY" in globals():
            self._groq_openai_client = OpenAI(api_key=GROQ_KEY, base_url=GROQ_URL)

        # Configure Gemini API globally and instantiate client
        if self._gemini_client is None and "GEMINI_API_KEY" in globals():
            self._gemini_client = genai.Client(api_key=GEMINI_API_KEY)

    def call(
        self,
        prompt_content: str,
        model: str,
        temperature: Optional[float],
        reasoning_effort: Optional[str],
        chain_of_thought: str,
        note_taking: str,
        max_retries: float = MAX_RETRIES,
        retry_delay: float = RETRY_DELAY,
        max_tokens: int = MAX_TOKENS,
    ) -> Tuple[str, str]:
        """
        Defines the API calls / simulations according to the configuration setup.
        Returns: notes, review.
        """

        self._maybe_init_clients()

        # --- Step 1: Generate notes ---

        # Construct final prompt basis
        note_taking_instruction = note_taking
        final_note_taking_prompt = note_taking_instruction + prompt_content + note_taking_instruction

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_note_taking_prompt += f" {chain_of_thought}"

        # Set provider for note-taking model
        provider_notes = "openai"
        if model in GROQ_MODELS:
            provider_notes = "groq"
        elif model in GEMINI_MODELS:
            provider_notes = "gemini"

        # Set temporary placeholder for the notes
        generated_notes = None

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial note
                if self.simulate:
                    generated_notes = (
                        f"This is a simulated note based on: Model='{model}', Temp='{temperature}', "
                        f"Effort='{reasoning_effort}', CoT='{chain_of_thought}', Notes='{note_taking}'. "
                    )
                    break
                # Call API
                else:
                    # For OpenAI or Groq
                    if provider_notes == "openai" or provider_notes == "groq":
                        kwargs = {}
                        # Set hyperparameter
                        if model in REASONING_MODELS and reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # Specify the user message
                        msgs = [{"role": "user", "content": final_note_taking_prompt}]

                        # Specify the client to use
                        client_to_use = self._groq_openai_client if provider_notes == "groq" else self._openai

                        # Call client and return the notes
                        resp = client_to_use.chat.completions.create(
                            model=model,
                            messages=msgs,
                            max_tokens=max_tokens,
                            **kwargs,
                        )
                        # Extract the notes
                        generated_notes = resp.choices[0].message.content or ""
                        break
                    # For Gemini
                    elif provider_notes == "gemini":

                        # Set and populate hyperparameter
                        gen_content_config_args = {}
                        gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                        # Call client and return the notes
                        resp = self._gemini_client.models.generate_content(
                            model=model,
                            contents=final_note_taking_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        # Extract the notes
                        generated_notes = resp.text
                        break

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Note Taking] provider={{provider_notes}} model={{model}} attempt={{attempt}} -> {{e}}", flush=True)
                if attempt == max_retries:
                    # Return failure string across both outputs
                    return NOTE_TAKING_FAILURE, REVIEW_GENERATION_FAILURE
                time.sleep(retry_delay)

        if generated_notes is None:
            # Return failure string across both outputs
            return NOTE_TAKING_FAILURE, REVIEW_GENERATION_FAILURE

        # --- Step 2: Generate the main review ---

        # Construct final prompt basis
        final_review_prompt = REVIEW_GENERATION + generated_notes + REVIEW_GENERATION

        # Add optional chain of thought instruction
        if chain_of_thought:
            final_review_prompt += f" {chain_of_thought}"

        # Set provider for review model
        provider_review = "openai"
        if model in GROQ_MODELS:
            provider_review = "groq"
        elif model in GEMINI_MODELS:
            provider_review = "gemini"

        for attempt in range(1, max_retries + 1):
            try:
                # Simulate artificial review
                if self.simulate:
                    generated_review = (
                        f"This is a simulated review: Model='{model}', Temp={temperature}, "
                        f"Effort='{reasoning_effort}', chain_of_thought='{chain_of_thought}'. "
                    )
                    return generated_notes, generated_review
                # Call API
                else:
                    # For OpenAI and Groq
                    if provider_review == "openai" or provider_review == "groq":
                        # Set hyperparameter
                        kwargs = {}
                        if model in REASONING_MODELS and reasoning_effort:
                            kwargs["reasoning"] = {"effort": reasoning_effort}
                        if temperature is not None:
                            kwargs["temperature"] = float(temperature)

                        # Specify user message
                        msgs = [{"role": "user", "content": final_review_prompt}]

                        # Specify the client to use
                        client_to_use = self._groq_openai_client if provider_review == "groq" else self._openai

                        # Call client and return the review
                        resp = client_to_use.chat.completions.create(
                            model=model,
                            messages=msgs,
                            max_tokens=max_tokens,
                            **kwargs,
                        )
                        # Extract the review
                        generated_review = resp.choices[0].message.content or ""

                    # For Gemini
                    elif provider_review == "gemini":

                        # Set and populate hyperparameter
                        gen_content_config_args = {}
                        gen_content_config_args["thinking_config"] = types.ThinkingConfig(thinking_level=reasoning_effort)

                        # Call client and return the review
                        resp = self._gemini_client.models.generate_content(
                            model=model,
                            contents=final_review_prompt,
                            config=types.GenerateContentConfig(**gen_content_config_args)
                        )
                        # Extract the review
                        generated_review = resp.text

                    return generated_notes, generated_review

            # Raise error flag in case of failure
            except Exception as e:
                print(f"[LLM ERROR - Review Generation] provider={{provider_review}} model={{model}} attempt={{attempt}} -> {{e}}", flush=True)
                if attempt == max_retries:
                    # Return notes and failure message for review generation
                    return generated_notes, REVIEW_GENERATION_FAILURE
                time.sleep(retry_delay)

        # Return notes and unknown error for review generation
        return generated_notes, UNKNOWN_ERROR_STATE

### Helper for output messages

In [ ]:
def _fmt_combo(combo: ParamCombo) -> str:
    """Helper to format the ParamCombo into a human-readable string"""
    parts = []
    parts.append(f"model={combo.model}")
    if combo.temperature is not None:
        parts.append(f"temperature={combo.temperature}")
    if combo.reasoning_effort is not None:
        parts.append(f"reasoning_effort={combo.reasoning_effort}")
    parts.append(f"chain_of_thought={combo.chain_of_thought}")
    parts.append(f"note_taking={combo.note_taking}")
    return ", ".join(parts)

def log_call(doc_id: str, combo: ParamCombo, **context: Dict[str, Any]):
    """Helper to enrich the ParamCombo string with information for output messages"""
    ctx = ", ".join(f"{k}={v}" for k, v in context.items() if v is not None)
    msg = f"[LLM CALL] paper={doc_id} | {_fmt_combo(combo)}"
    if ctx:
        msg += f" | {ctx}"
    print(msg, flush=True)

### Simulation definition

In [ ]:
def run_parameter_combo(
    combo: ParamCombo,
    review_target: Dict,
    client: Optional[LLMClient] = None,
) -> pd.DataFrame:
    """
    Sets id, content and parameters for the client call.
    Calls helper function to print the information.
    Calls the client and returns the results in single data frame row.
    """
    # If client specified then use client, otherwise fake client for test
    client = client or LLMClient(simulate=True)

    # Extract the current content
    paper_content = review_target['parsed_text']

    # Extract the current id
    doc_id = review_target["paper_id"]

    # Initialize a dictionary for the current id and parameters
    current_paper_record = {
        "paper_id": doc_id,
        "model": combo.model,
        "temperature": combo.temperature,
        "reasoning_effort": combo.reasoning_effort,
        "chain_of_thought": combo.chain_of_thought,
        "note_taking": combo.note_taking
    }

    # Loop to generate multiple notes/reviews
    for review_idx in range(NUM_REVIEWS_PER_PAPER):

        # Include informative output message
        log_call(doc_id, combo, review_number=review_idx + 1)

        # Parse current content and parameters to the client
        notes, review = client.call(
            model=combo.model,
            prompt_content=paper_content,
            temperature=(combo.temperature if combo.model not in REASONING_MODELS else None),
            reasoning_effort=(combo.reasoning_effort if combo.model in REASONING_MODELS else None),
            chain_of_thought=combo.chain_of_thought,
            note_taking=combo.note_taking
        )

        # Store notes
        current_paper_record[f"llm_notes_{review_idx + 1}"] = notes_content
        # Store review
        current_paper_record[f"llm_review_{review_idx + 1}"] = review

    # Convert the record into a DataFrame row
    df = pd.DataFrame(current_paper_record, index=[0])
    return df

### Configuration grid

In [ ]:
# Complete list of models
MODELS = OPENAI_MODELS + GROQ_MODELS + GEMINI_MODELS

# Generate every single combination
raw_grid = list(itertools.product(MODELS, TEMPS, REASONING, CoT, NOTE_TAKING))
grid_df = pd.DataFrame(raw_grid, columns=["model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"])

# Conditional deletion whether a model has a temperature or reasoning parameter
grid_df["reasoning_effort"] = np.where(
    ~grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["reasoning_effort"]
)
grid_df["temperature"] = np.where(
    grid_df["model"].isin(REASONING_MODELS), np.nan, grid_df["temperature"]
)

# Drop duplicates
experiment_config = grid_df.drop_duplicates(ignore_index=True)

print(f"Total valid parameter combos: {len(experiment_config)}")

display(experiment_config)

### Helper for logging

In [ ]:
def _nan_to_none(x):
    """Use pandas' isna to catch float('nan'), numpy.nan, pd.NA, and None"""
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    return x


def combo_key_tuple(row) -> tuple:
    """Combines the congfigurational setting with None instead of NaN"""
    return (
        row["model"],
        _nan_to_none(row["temperature"]),
        _nan_to_none(row["reasoning_effort"]),
        row["chain_of_thought"],
        row["note_taking"]
    )

def combo_key_str(row) -> str:
    """Creates readable combo string for logging"""
    t = combo_key_tuple(row)
    return f"model={t[0]}|temp={t[1]}|re={t[2]}|cot={t[3]}|notes={t[4]}"

### LLM review generation

In [ ]:
if __name__ == '__main__':

    print(f"Processing and saving results to: {RESULTS_PATH}", flush = True)
    print(f"Logging successful iterations to: {LOG_PATH}", flush = True)

    # Initialize client
    client = LLMClient(simulate=SIMULATION_ACTIVE, seed=SEED_VALUE)

### --- 1. Check for already existing logs and results --- ###

    # Initialize keys
    done_keys = set()

    # Check and read successful paper-parameter combos from log file
    if os.path.exists(LOG_PATH):
        try:
            log_df = pd.read_parquet(LOG_PATH)
            # Check existence and load content from the papers id and completion status
            if 'paper_id' in log_df.columns and 'completed_param_combo' in log_df.columns:
                done_keys.update(zip(log_df['paper_id'].tolist(), log_df['completed_param_combo'].tolist()))
                print(f"Loaded {len(done_keys)} completed paper-parameter combo entries from log.")
            else:
                print(f"Warning: '{LOG_PATH}' exists but does not contain 'paper_id' and 'completed_param_combo' columns. Starting with empty log.")

        except Exception as e:
            print(f"Error reading existing log parquet file: {e}. Starting with empty log.")
            done_keys = set()

    # Check and read successful paper-parameter combos from results file
    if os.path.exists(RESULTS_PATH):
        try:
            result_df = pd.read_parquet(RESULTS_PATH)
            if not result_df.empty:
                # Set all columns that define a unique paper-parameter combo
                needed_col_for_grouping = ["paper_id", "model", "temperature", "reasoning_effort", "chain_of_thought", "note_taking"]

                result_df_copy = result_df.copy()

                # Ensure no nessecary column is missing (they should not)
                missing_cols = [c for c in needed_col_for_grouping if c not in result_df_copy.columns]

                if not missing_cols:
                    # Normalize NAs to None for keys (only for relevant columns)
                    for c in ["temperature", "reasoning_effort"]:
                        result_df_copy[c] = result_df_copy[c].where(pd.notna(result_df_copy[c]), None)

                    # Count amount of reviews per unqiue combo
                    paper_combo_counts = (
                        result_df_copy.groupby(needed_col_for_grouping)
                        .size()
                        .reset_index(name='review_count')
                    )

                    # Identify paper-configuration combos that have the expected number of reviews
                    for _, r in paper_combo_counts.iterrows():
                        if int(r["review_count"]) == NUM_REVIEWS_PER_PAPER:
                            # Load `paper_id` and `completed_configuration_str` into the keys
                            completed_configuration_str = combo_key_str(r)
                            done_keys.add((r['paper_id'], completed_configuration_str))
        except Exception as e:
            print(f"Error reading existing results parquet file: {e}. Starting with empty results.")

    print(f"Already-completed paper-configuration combos: {len(done_keys)}")

    ### --- 2. Create notes and reviews for all paper-parameter combos --- ###

    # Outer loop: iterate over parameter combinations
    for idx_combo, row_combo in tqdm(experiment_config.iterrows(), total=len(experiment_config), desc="Processing Combos"):

        # Set string representation of the current combo
        param_combo_str_current = combo_key_str(row_combo)

        # Inner loop: iterate over each paper
        for idx_paper, paper_data in enumerate(targets):
            paper_id = paper_data["paper_id"]
            paper_combo_key = (paper_id, param_combo_str_current)

            # Check if this specific paper-combo is already completed
            if paper_combo_key in done_keys:
                print(f"[SKIP] Paper {paper_id} with combo {param_combo_str_current} already complete.")
                continue

            # Extract current parameters from the configuration grid
            combo = ParamCombo(
                model=row_combo["model"],
                temperature=None if pd.isna(row_combo["temperature"]) else float(row_combo["temperature"]),
                reasoning_effort=None if pd.isna(row_combo["reasoning_effort"]) else str(row_combo["reasoning_effort"]),
                chain_of_thought=row_combo["chain_of_thought"],
                note_taking=row_combo["note_taking"]
            )

            print(f"\n[RUN] Combo {idx_combo+1}/{len(experiment_config)}, Paper {idx_paper+1}/{len(targets)} -> {paper_combo_key[0]} | {paper_combo_key[1]}", flush=True)

            try:
                # Make actual simulation/call for this single paper
                df_combo_paper = run_parameter_combo(combo, paper_data, client=client)

                # Append results to RESULTS_PATH
                if os.path.exists(RESULTS_PATH):
                    # Read, concat and save
                    existing_df = pd.read_parquet(RESULTS_PATH)
                    combined_df = pd.concat([existing_df, df_combo_paper], ignore_index=True)
                    combined_df.to_parquet(RESULTS_PATH, index=False)
                else:
                    # Write a new result file
                    df_combo_paper.to_parquet(RESULTS_PATH, index=False)

                # Add done keys as soon as result is stored (liberal)
                done_keys.add(paper_combo_key)

                # Log successful paper-parameter combo
                new_log_entry_df = pd.DataFrame(
                    {
                        'paper_id': paper_id,
                        'completed_param_combo': param_combo_str_current
                    },
                    index=[0]
                )

                try:
                    if os.path.exists(LOG_PATH):
                        # Read, concat and save
                        existing_log_df = pd.read_parquet(LOG_PATH)
                        combined_log_df = pd.concat([existing_log_df, new_log_entry_df], ignore_index=True)
                        combined_log_df.to_parquet(LOG_PATH, index=False)
                    else:
                        # Write a new log file
                        new_log_entry_df.to_parquet(LOG_PATH, index=False)

                except Exception as log_e:
                    print(f"Warning: Could not append to existing log parquet file {LOG_PATH}.", flush=True)

            # Catch a code error (before the result is stored and the key is added)
            except Exception as e:
                print(f"[ERROR] Paper {paper_id} with combo {param_combo_str_current} -> {type(e).__name__}: {e}", flush=True)

In [ ]:
# load result.parquet-file
if os.path.exists(RESULTS_PATH):
    try:
        llm_results_df = pd.read_parquet(RESULTS_PATH)
        # Check results only if loading was successful
        display(llm_results_df.head(5))
        display(llm_results_df.shape)
    except:
        print(f"Warning: Could not read existing results parquet file {RESULTS_PATH}.")
else:
    print(f"Results file '{RESULTS_PATH}' does not exist.")

# load log.parquet-file
if os.path.exists(LOG_PATH):
    try:
        llm_log_df = pd.read_parquet(LOG_PATH)
        # Check log only if loading was successful
        display(llm_log_df.head(5))
        display(llm_log_df.shape)
    except:
        print(f"Warning: Could not read existing log parquet file {LOG_PATH}.")
else:
    print(f"Log file '{LOG_PATH}' does not exist.")